# Toric code from boundary maps

This notebook constructs a small toric CSS code from a periodic square cellulation. Physical qubits are placed on edges, so the relevant chain complex is

$$
C_2 \xrightarrow{\partial_2} C_1 \xrightarrow{\partial_1} C_0.
$$

The CSS check matrices are

$$
H_X=\partial_1,
\qquad
H_Z=\partial_2^T,
$$

and their commutation follows from

$$
H_XH_Z^T=\partial_1\partial_2=0.
$$

In [ ]:
import numpy as np

from homoloqode import square_toric_complex
from homoloqode.algebra import matmul

## 1. Periodic square cellulation

For a lattice of linear size $L$, the square cellulation has $L^2$ vertices, $2L^2$ edges, and $L^2$ faces. The periodic identifications make the underlying surface a torus.

In [ ]:
size = 3
cell_complex = square_toric_complex(size)

print(f"vertices: {len(cell_complex.vertices)}")
print(f"edges / physical qubits: {len(cell_complex.edges)}")
print(f"faces: {len(cell_complex.faces)}")
print("first four edge labels:", cell_complex.edge_ids[:4])

Each horizontal and vertical edge is a distinct one-cell. This remains true when two edges share endpoints, which is essential for periodic lattices.

In [ ]:
chain_complex = cell_complex.to_chain_complex()
d1 = chain_complex.d1
d2 = chain_complex.d2

print("d1 shape (vertices x edges):", d1.shape)
print("d2 shape (edges x faces):", d2.shape)
print("d1 =")
print(d1)
print("d2 =")
print(d2)

## 2. Chain-complex validation

Every face boundary must itself have zero boundary. The implementation validates this identity when constructing `ChainComplex2D`.

In [ ]:
boundary_of_boundary = matmul(d1, d2)

print("d1 @ d2 =")
print(boundary_of_boundary)
assert not np.any(boundary_of_boundary)

## 3. Homological CSS code

The basis elements of $C_1$ become physical qubits. Vertex-edge incidence gives the $X$ checks, and transposed edge-face incidence gives the $Z$ checks.

In [ ]:
code = chain_complex.to_css_code()
hx = code.hx
hz = code.hz

print("H_X shape:", hx.shape)
print("H_Z shape:", hz.shape)
print("rank(H_X):", code.rank_x)
print("rank(H_Z):", code.rank_z)
print(f"[[n, k]] = [[{code.n}, {code.k}]]")
print("dim H_1:", chain_complex.betti_1)

In [ ]:
css_commutator = matmul(hx, hz.T)

print("H_X @ H_Z.T =")
print(css_commutator)
assert not np.any(css_commutator)
assert code.n == 18
assert code.k == 2

There are nine vertex checks and nine face checks, but each family contains one global linear dependency. Consequently both check matrices have rank eight and the code encodes two logical qubits.

In [ ]:
print("First three X stabilizers:")
for label, pauli in list(zip(code.x_check_labels, code.x_stabilizer_strings()))[:3]:
    print(f"  {label}: {pauli}")

print("\nFirst three Z stabilizers:")
for label, pauli in list(zip(code.z_check_labels, code.z_stabilizer_strings()))[:3]:
    print(f"  {label}: {pauli}")

## 4. Single-qubit errors and syndromes

A $Z$ error on one edge anticommutes with the two adjacent vertex checks. An $X$ error on the same edge anticommutes with the two adjacent face checks.

In [ ]:
edge_label = "e_h[0,0]"
edge_index = code.qubit_labels.index(edge_label)
single_edge_error = np.zeros(code.n, dtype=np.uint8)
single_edge_error[edge_index] = 1

z_error_syndrome = code.syndrome(z_error=single_edge_error)
x_error_syndrome = code.syndrome(x_error=single_edge_error)

violated_x_checks = [
    label
    for label, bit in zip(code.x_check_labels, z_error_syndrome.x_checks)
    if bit
]
violated_z_checks = [
    label
    for label, bit in zip(code.z_check_labels, x_error_syndrome.z_checks)
    if bit
]

print(f"Z error on {edge_label} violates X checks:", violated_x_checks)
print(f"X error on {edge_label} violates Z checks:", violated_z_checks)
assert len(violated_x_checks) == 2
assert len(violated_z_checks) == 2

## 5. Paired logical operators

Logical $Z$ representatives are obtained from

$$
\ker(H_X)/\operatorname{row}(H_Z),
$$

while logical $X$ representatives come from

$$
\ker(H_Z)/\operatorname{row}(H_X).
$$

The returned bases are transformed so that $X_i\cdot Z_j=\delta_{ij}$.

In [ ]:
logicals = code.logical_basis()
pairing = matmul(logicals.x, logicals.z.T)

for index in range(logicals.count):
    x_support = [
        label for label, bit in zip(code.qubit_labels, logicals.x[index]) if bit
    ]
    z_support = [
        label for label, bit in zip(code.qubit_labels, logicals.z[index]) if bit
    ]
    print(f"logical X_{index} support:", x_support)
    print(f"logical Z_{index} support:", z_support)

print("\nlogical pairing X @ Z.T =")
print(pairing)
assert np.array_equal(pairing, np.eye(code.k, dtype=np.uint8))

## Current boundary of the implementation

The notebook demonstrates the complete algebraic path from a periodic cellulation to a valid CSS code, stabilizers, syndromes, and paired logical representatives. Exact distance computation, decoding, geometric plotting, and Qiskit syndrome-extraction circuits are the next MVP stages.